## 1. Configuración Inicial de Parámetros

Ejecuta la siguiente celda para configurar los parámetros de auditoría. 
Todos los valores están documentados con comentarios inline para facilitar inspección antes del envío.

In [1]:
from pathlib import Path
import sys

# Localizar el soporte tanto desde la carpeta del cuaderno como desde el repo.
SUPPORT_DIR = next((root for root in (Path.cwd(), Path.cwd() / "auditoria_IA",
    Path.cwd() / "plan_v2" / "auditoria_IA") if (root / "auditoria_config.py").is_file()), None)
if SUPPORT_DIR is None:
    raise FileNotFoundError("Abre el cuaderno desde su carpeta o desde la raiz del repositorio.")
sys.path.insert(0, str(SUPPORT_DIR.resolve()))
from auditoria_config import AuditConfig

# Cambia solo esta ruta para auditar PDF, TXT, MD o DOCX; no enviar codigo LaTeX crudo.
INPUT_DOCUMENT = SUPPORT_DIR.parent / "plan_tesis_cables_pinn.pdf"
OUTPUT_DIR = SUPPORT_DIR / "auditoria_resultados"
API_URL = "http://127.0.0.1:8000"  # Solo localhost; el documento no se envia a una nube.
CONTAINER_NAME = "sloptotal"
START_CONTAINER = True  # Arranca el existente; no recrea ni elimina contenedores/volumenes.
STARTUP_TIMEOUT = 60

ANALYSIS_MODE = "both"  # whole, chunks o both; usa el mismo endpoint para ambos.
CHUNK_MODE = "structure"  # Respeta encabezados/parrafos; words usa ventanas por palabras.
MAX_WORDS = 400  # Compromiso de contexto y coste; no es una garantia de calibracion.
MAX_CHARS = 5000  # Limite adicional; nunca se fragmenta una palabra.
MAX_CHUNKS = None  # None revisa todos; un limite produce una auditoria parcial.
REVIEW_THRESHOLD = 65.0  # Prioridad de lectura, no porcentaje de autoria IA.
REQUEST_TIMEOUT = 180  # Espera de cada POST; no reintentar tras timeout automaticamente.
REPORT_TIMEOUT = 300  # Espera adicional de un ticket, si el servidor lo devuelve.
STOP_AFTER_TIMEOUTS = 1  # Evitar acumular trabajos que sigan activos en el servidor.
DRY_RUN = False  # True prueba extraccion/informe sin red ni Docker.
SMOKE_TEST = True  # Inicio seguro: global + fragmento de una muestra, NO del PDF completo.
SMOKE_CHUNKS = 1  # Para auditoria completa cambia SMOKE_TEST a False.

config = AuditConfig(
    input_path=INPUT_DOCUMENT, output_dir=OUTPUT_DIR, api_url=API_URL,
    container_name=CONTAINER_NAME, start_container=START_CONTAINER,
    startup_timeout=STARTUP_TIMEOUT, mode=ANALYSIS_MODE, chunk_mode=CHUNK_MODE,
    max_words=MAX_WORDS, max_chars=MAX_CHARS, max_chunks=MAX_CHUNKS,
    request_timeout=REQUEST_TIMEOUT, report_timeout=REPORT_TIMEOUT,
    review_threshold=REVIEW_THRESHOLD, dry_run=DRY_RUN,
    stop_after_timeouts=STOP_AFTER_TIMEOUTS,
)
print(f"Entrada: {config.input_path}")
print(f"Modo: {config.mode}; fragmentacion: {config.chunk_mode}; prueba de humo: {SMOKE_TEST}")
print(f"Salida: {config.output_dir}")

Entrada: c:\usr\ths_mia_fiis\tesis_MIA_lk\plan_v2\plan_tesis_cables_pinn.pdf
Modo: both; fragmentacion: structure; prueba de humo: True
Salida: c:\usr\ths_mia_fiis\tesis_MIA_lk\plan_v2\auditoria_IA\auditoria_resultados


## 2. Importaciones y Verificación de Entorno

In [2]:
import json
from dataclasses import replace
from auditoria_config import prepare_document, diagnose_service, run_audit
from documentos import TextBlock
from auditar_plan import get_json

print(f"Python: {sys.version.split()[0]} ({sys.executable})")
print(f"Soporte: {SUPPORT_DIR.resolve()}")
print("Formatos: PDF con texto, TXT, Markdown y DOCX; un PDF escaneado requiere OCR previo.")

Python: 3.12.9 (c:\Python312\python.exe)
Soporte: C:\usr\ths_mia_fiis\tesis_MIA_lk\plan_v2\auditoria_IA
Formatos: PDF con texto, TXT, Markdown y DOCX; un PDF escaneado requiere OCR previo.


## 3. Diagnóstico de Servicio

Verifica el estado de Docker y la API sloptotal. En `DRY_RUN=True`, este paso devuelve None sin efectos colaterales.

In [3]:
diagnostics = None
if config.dry_run:
    print("Modo seco: sin solicitudes de red ni arranque de Docker.")
else:
    diagnostics = diagnose_service(config)
    docker = diagnostics.get("docker", {})
    health = diagnostics.get("health")
    print(f"Contenedor existente: {docker.get('exists', False)}")
    print(f"Estado registrado: {docker.get('state', {}).get('Status', 'desconocido')}")
    if docker.get("start_output"):
        print(f"Arranque: {docker['start_output']}")
    if health:
        engines = health.get("engines", {})
        print(f"API: {health.get('status', 'desconocido')}; motores: {engines.get('loaded', '?')}/{engines.get('total', '?')}")
        print(f"Motores fallidos: {', '.join(engines.get('failed', [])) or 'ninguno reportado'}")
        diagnostics["queue"] = get_json(config.api_url, "/api/queue/status")
        print("Colas:", json.dumps(diagnostics["queue"], ensure_ascii=False))
    else:
        raise RuntimeError(f"API no disponible: {docker.get('health_error', docker.get('error', 'sin detalles'))}")

sloptotal: 12/23 motores cargados.
Contenedor existente: True
Estado registrado: running
API: healthy; motores: 12/23
Motores fallidos: perplexity, cross_perplexity, classifier_openai, classifier_chatgpt, classifier_remodetect, classifier_fakespot, classifier_e5, classifier_tmr, classifier_bert_raid, classifier_desklib, classifier_superannotate
Colas: {"snippet": {"active": 0, "max_concurrent": 4, "queued": 0, "max_queue": 50, "accepting": true, "avg_latency_ms": 1440940.0}, "quick": {"active": 0, "max_concurrent": 4, "queued": 0, "max_queue": 50, "accepting": true, "avg_latency_ms": 498875.3}, "full": {"active": 0, "max_concurrent": 2, "queued": 0, "max_queue": 20, "accepting": true, "avg_latency_ms": 3435796.6}, "short": {"active": 0, "max_concurrent": 2, "queued": 0, "max_queue": 20, "accepting": true, "avg_latency_ms": 13432103.3}}
Contenedor existente: True
Estado registrado: running
API: healthy; motores: 12/23
Motores fallidos: perplexity, cross_perplexity, classifier_openai, cl

## 4. Extracción y Vista Previa del Documento

In [5]:
full_document, chunks = prepare_document(config)
document = full_document
print(f"Formato: {document.metadata.get('format', config.input_path.suffix)}")
print(f"Texto: {len(document.text.split()):,} palabras; {len(document.text):,} caracteres")
print(f"Fragmentos seleccionados: {len(chunks)}/{document.metadata.get('audit_total_chunks', len(chunks))}")
for warning in document.warnings:
    print(f"Advertencia: {warning}")

if SMOKE_TEST:
    if SMOKE_CHUNKS < 1 or not chunks:
        raise ValueError("La prueba necesita al menos un fragmento.")
    candidates = [chunk for chunk in chunks if len(chunk.text.split()) >= 150]
    if not candidates:
        candidates = sorted(chunks, key=lambda chunk: len(chunk.text), reverse=True)
    chunks = candidates[:SMOKE_CHUNKS]
    sample = "\n\n".join(chunk.text for chunk in chunks)
    if len(sample.strip()) < 50:
        raise ValueError("Muestra demasiado corta para el endpoint (menos de 50 caracteres).")
    document = replace(full_document, text=sample,
        blocks=[TextBlock(sample, "muestra de prueba de humo")],
        metadata={**full_document.metadata, "smoke_test": True,
            "source_document_words": len(full_document.text.split())},
        warnings=[*full_document.warnings, "PRUEBA DE HUMO: global corresponde SOLO a la muestra, no al documento completo."])
    print(f"PRUEBA DE HUMO: {len(sample.split())} palabras; {len(chunks)} fragmento(s). Global = muestra.")

if chunks:
    first = chunks[0]
    print(f"Primer fragmento: {first.chunk_id}; ubicaciones: {', '.join(first.locations)}")
    print(f"Secciones: {', '.join(first.sections) or 'no detectadas'}")
    print(first.text[:500])
assert all(len(chunk.text.split()) <= config.max_words and len(chunk.text) <= config.max_chars for chunk in chunks)
print("Limites de fragmentacion comprobados.")

Formato: pdf
Texto: 24,921 palabras; 161,838 caracteres
Fragmentos seleccionados: 208/208
PRUEBA DE HUMO: 174 palabras; 1 fragmento(s). Global = muestra.
Primer fragmento: c0009; ubicaciones: page_2_block_9, page_2_block_10, page_2_block_11, page_2_block_12, page_2_block_13, page_2_block_14, page_2_block_15, page_2_block_16, page_2_block_17, page_2_block_18, page_2_block_19, page_2_block_20, page_2_block_21, page_2_block_22, page_3_block_23
Secciones: DATOS GENERALES
DATOS GENERALES

Elemento
Descripción

Título del plan
DISEÑO Y EVALUACIÓN DE UNA RED NEURONAL IN-
FORMADA POR FÍSICA PARA ESTIMAR LA TEMPERATU-
RA Y LA AMPACIDAD DE CABLES ELÉCTRICOS ENTE-
RRADOS EN ENTORNOS HETEROGÉNEOS.

Tesistas
Luis Enrique Koc Góngora;
Herbert Antonio Meléndez García.

Asesor
Samuel Oporto.

Programa
Maestría en Inteligencia Artificial.

Unidad académica
Unidad de Posgrado de la Facultad de Ingeniería Industrial
y de Sistemas, Universidad Nacional de Ingeniería.

Área invo
Limites de fragmentacion co

## 5. Ejecución de Auditoría

Se enviarán requests a la API según los parámetros configurados. 
El progreso se mostrará en tiempo real.

In [6]:
print(f"Iniciando: modo={config.mode}; fragmentacion={config.chunk_mode}; prueba de humo={SMOKE_TEST}")
print("El porcentaje indica tareas terminadas (incluye errores/omisiones), no porcentaje de texto humano o IA.")
report_path = run_audit(config, document, chunks, diagnostics)
results = json.loads((report_path.parent / "results.json").read_text(encoding="utf-8"))
scored = sum(item.get("score") is not None for item in results)
failed = sum(item["status"] in {"error", "timeout", "no_score"} for item in results)
skipped = sum(item["status"] == "skipped" for item in results)
print(f"Resumen: {scored}/{len(results)} tareas con puntuacion; {failed} errores/sin score; {skipped} omitidas.")
if config.dry_run:
    print("Validacion local solamente; no se solicito puntuacion.")
elif scored != len(results):
    print("AUDITORIA INCOMPLETA: revisar errores; disponer de informe no demuestra que la API funciono.")
elif SMOKE_TEST:
    print("PRUEBA DE HUMO CORRECTA: ambos resultados corresponden solo a la muestra seleccionada.")
else:
    print("Todas las tareas seleccionadas obtuvieron puntuacion; comprobar cobertura y limitaciones.")
assert report_path.is_file()
print(f"Informe Markdown: {report_path}")

Iniciando: modo=both; fragmentacion=structure; prueba de humo=True
El porcentaje indica tareas terminadas (incluye errores/omisiones), no porcentaje de texto humano o IA.
[0/2 0.0%] Inicio: 174 palabras; 1 fragmentos seleccionados.
  Enviando global (174 palabras) a /api/analyze...


[1/2 50.0%] global: timeout, sin puntuacion. Muchos saltos de linea: comprobar tablas, formulas y orden de extraccion antes de interpretar.
  HTTPConnectionPool(host='127.0.0.1', port=8000): Read timed out. (read timeout=180)
[2/2 100.0%] c0009: skipped, sin puntuacion. Muchos saltos de linea: comprobar tablas, formulas y orden de extraccion antes de interpretar.
  HTTPConnectionPool(host='127.0.0.1', port=8000): Read timed out. (read timeout=180)
[2/2 100.0%] c0009: skipped, sin puntuacion. Muchos saltos de linea: comprobar tablas, formulas y orden de extraccion antes de interpretar.
  Se detuvieron nuevos envios despues de timeouts; una solicitud anterior puede seguir activa en el servidor.
Finalizado: 0/2 tareas con puntuacion. Informe: c:\usr\ths_mia_fiis\tesis_MIA_lk\plan_v2\auditoria_IA\auditoria_resultados\run_20261114T043456_577496Z\auditoria_resultado.md
Resumen: 0/2 tareas con puntuacion; 1 errores/sin score; 1 omitidas.
AUDITORIA INCOMPLETA: revisar errores; disponer de info

## 6. Visualización de Reporte

Se muestra el reporte Markdown completo con resultados y análisis.

In [7]:
from IPython.display import Markdown, display

if not report_path.is_file():
    raise FileNotFoundError(f"No se genero el informe: {report_path}")
report_content = report_path.read_text(encoding="utf-8")
display(Markdown(report_content))

# Auditoria de documento con sloptotal

Fecha UTC: 2026-11-14T04:37:56.623208+00:00
Documento: c:\usr\ths_mia_fiis\tesis_MIA_lk\plan_v2\plan_tesis_cables_pinn.pdf

## Alcance y limitaciones

Tareas seleccionadas: 2. Con puntuacion: 0. Con error/sin puntuacion: 1.
Fragmentos seleccionados: 1 de 208.
La puntuacion 0-100 del detector NO es el porcentaje de palabras creadas por IA ni prueba de autoria.
Documento global y fragmentos usan /api/analyze; sus puntuaciones no se suman ni son directamente equivalentes.
Los comentarios editoriales son heuristicas independientes: no explican causalmente la puntuacion.
No se modifica ni se reescribe la fuente. No hay calibracion propia para espanol academico ni umbral garantizado.

## Servicio

Motores cargados: 12/23.
Motores fallidos: perplexity, cross_perplexity, classifier_openai, classifier_chatgpt, classifier_remodetect, classifier_fakespot, classifier_e5, classifier_tmr, classifier_bert_raid, classifier_desklib, classifier_superannotate.
Una cobertura incompleta puede cambiar la puntuacion respecto a una instalacion completa.

## Extraccion

Formato: pdf. Palabras: 174. Caracteres: 1246.
- PRUEBA DE HUMO: global corresponde SOLO a la muestra, no al documento completo.

## Resultados

| Tarea | Ubicacion | Estado | Score 0-100 | Atencion |
| --- | --- | --- | --- | --- |
| global | documento completo | timeout | no disponible | sin prioridad por score |
| c0009 | page_2_block_9, page_2_block_10, page_2_block_11, page_2_block_12, page_2_block_13, page_2_block_14, page_2_block_15, page_2_block_16, page_2_block_17, page_2_block_18, page_2_block_19, page_2_block_20, page_2_block_21, page_2_block_22, page_3_block_23 | skipped | no disponible | sin prioridad por score |

## global

Estado: timeout. Score: no disponible.
Secciones: no detectadas.
Error: HTTPConnectionPool(host='127.0.0.1', port=8000): Read timed out. (read timeout=180)
- Muchos saltos de linea: comprobar tablas, formulas y orden de extraccion antes de interpretar.

## c0009

Estado: skipped. Score: no disponible.
Secciones: DATOS GENERALES.
Error: Se detuvieron nuevos envios despues de timeouts; una solicitud anterior puede seguir activa en el servidor.
- Muchos saltos de linea: comprobar tablas, formulas y orden de extraccion antes de interpretar.

Texto analizado:

> DATOS GENERALES
> 
> Elemento
> Descripción
> 
> Título del plan
> DISEÑO Y EVALUACIÓN DE UNA RED NEURONAL IN-
> FORMADA POR FÍSICA PARA ESTIMAR LA TEMPERATU-
> RA Y LA AMPACIDAD DE CABLES ELÉCTRICOS ENTE-
> RRADOS EN ENTORNOS HETEROGÉNEOS.
> 
> Tesistas
> Luis Enrique Koc Góngora;
> Herbert Antonio Meléndez García.
> 
> Asesor
> Samuel Oporto.
> 
> Programa
> Maestría en Inteligencia Artificial.
> 
> Unidad académica
> Unidad de Posgrado de la Facultad de Ingeniería Industrial
> y de Sistemas, Universidad Nacional de Ingeniería.
> 
> Área involucrada
> Inteligencia artificial científica, modelado computacional y
> sistemas eléctricos de potencia.
> 
> Lugar de desarrollo
> Entorno computacional de la Facultad de Ingeniería Indus-
> trial y de Sistemas de la Universidad Nacional de Ingeniería
> (FIIS-UNI) y casos de referencia documentados de instala-
> ciones de cables eléctricos enterrados.
> 
> Duración estimada
> Doce meses, desde julio de 2026 hasta junio de 2027.
> 
> Fecha de inicio
> Julio de 2026.
> 
> Fecha de culminación
> prevista
> Junio de 2027.
> 
> Tipo de graduación
> Maestría de especialización o profesionalizante.
> 
> Artefacto de
> investigación
> Modelo y prototipo computacional 2D basado en redes neu-
> ronales informadas por física para analizar campos térmicos
> y ampacidad en entornos heterogéneos.
> 
> ııı

## Metodo de revision humana

1. Verificar el texto extraido y sus ubicaciones contra el documento original.
2. Leer los fragmentos priorizados con su contexto; conservar citas, formulas y significado tecnico.
3. Revisar claridad, respaldo documental y consistencia, sin alterar estilo solo para cambiar un score.
4. Registrar decisiones humanas y versiones. Evaluar falsos positivos con textos humanos comparables, no con un score aislado.

Configuracion, hashes, texto extraido y respuestas originales estan junto a este informe.


# Auditoria reproducible con sloptotal local

Entrada generica: PDF con texto, TXT, Markdown o DOCX. El mismo lector y ejecutor sirven para otros documentos; la ruta del plan es solo el valor inicial del cuaderno. Un PDF escaneado requiere OCR previo; `.doc` antiguo requiere conversion a `.docx`.

## Ejecucion

1. Revisar la primera celda: ruta, `SMOKE_TEST`, `DRY_RUN`, modo global/fragmentos, limites y tiempos de espera.
2. Ejecutar las celdas en orden. Por defecto la prueba de humo analiza una muestra; su resultado global NO representa el documento completo.
3. Con `SMOKE_TEST=False` y `MAX_CHUNKS=None`, se selecciona todo el documento y todos sus fragmentos. Se conserva el texto extraible, incluidas portada, indices y referencias: no se eliminan por reglas especificas del plan.
4. Cada tarea imprime el porcentaje de tareas terminadas, puntuacion si existe y una observacion editorial. El porcentaje incluye errores y omisiones, no es porcentaje de autoria ni avance interno del servidor.
5. Revisar el Markdown y los JSON de cada ejecucion. Un informe generado sin scores NO equivale a una auditoria realizada con exito.

## Limites y diagnostico

- El mismo endpoint `/api/analyze` recibe texto global y fragmentos secuencialmente. Sus puntuaciones no se suman ni equivalen al porcentaje de palabras escritas por IA.
- Estructura y limites de palabras/caracteres controlan la segmentacion, con ubicaciones recuperables en el original.
- Los comentarios editoriales son heuristicas independientes, no una explicacion causal de las decisiones del detector.
- Docker solo inicia un contenedor existente. No recrea imagenes, no borra volumenes y no altera TLS.
- `/health` saludable no garantiza un analisis operativo. La prueba real encontro 12/23 motores y una espera de 180 s agotada, con reintentos de descarga de `gpt2-medium` y `CERTIFICATE_VERIFY_FAILED` en el contenedor.
- Tras un timeout, la solicitud puede seguir activa en el servidor: no repetir el envio sin revisar la cola. Antes de una auditoria completa hay que resolver los certificados y el acceso autorizado a los modelos, sin desactivar TLS.

No se modifica el documento fuente. Una alerta exige revision humana con contexto, citas e historial de versiones; no establece autoria.